# Taller MEDIQA-CORR: detección y corrección de errores médicos (versión corregida)

Cada texto clínico es correcto o contiene **un** error. El sistema debe: **A)** decir si hay error, **B)** indicar la oración errónea y **C)** proponer la oración corregida.

## Pipeline

```
CSV -> agrupar casos -> split train/dev/test por grupo -> oraciones
    -> Detector (Bio_ClinicalBERT) -> flag (A) y oración (B)
    -> Corrector (Flan-T5) -> corrección (C) -> evaluación (métrica oficial) -> archivo de envío
```

| Caja | Modelo o herramienta | Función en la tarea |
|---|---|---|
| Datos | pandas, TF-IDF | Carga, segmentación oficial y agrupación de versiones del mismo caso. |
| Detector | `emilyalsentzer/Bio_ClinicalBERT` + cabeza de clasificación | Da una probabilidad de error a cada oración. De ahí salen A y B. |
| Corrector | `google/flan-t5-base` afinado | Reescribe la oración errónea. Produce C. |
| Evaluación | `rouge`, `bert_score` | A y B con exactitud; C con ROUGE-1 y BERTScore configurados como el script oficial. |

## Cambios frente a la versión anterior
1. **Fuga de datos entre train y validación.** El MS se construyó inyectando en cada pregunta de MedQA una respuesta correcta y otra incorrecta, así que casi todos los casos aparecen dos veces (una versión correcta y una con error). Partir por `Text ID` dejaba al 71 % de los textos de validación con su gemelo en entrenamiento. Ahora se agrupan los textos casi idénticos (coseno TF-IDF > 0,7) y se parte **por grupo**.
2. **Umbral elegido y evaluado en el mismo conjunto.** Ahora hay tres conjuntos: train, dev (para el umbral) y test (solo para reportar).
3. **BERTScore se caía** con `deberta-xlarge-mnli` en transformers 5 (`OverflowError` por `model_max_length` infinito) y con predicciones vacías. Corregido.
4. **Métrica de C alineada con el script oficial**: paquete `rouge` (ROUGE-1 F), BERTScore con `rescale_with_baseline=True` y recorte a [0, 1].
5. **Tokenizador de Bio_ClinicalBERT**: el modelo se preentrenó *cased* (`--do_lower_case=False`), pero Hugging Face lo carga en minúsculas por defecto. Ahora se carga con `do_lower_case=False`.
6. **Línea base de B**: "nunca hay error" (-1 en todo) da más exactitud en B que "siempre la última". Se agregó.
7. **Formato de envío oficial** (`text_id flag sent_id corrección`) además del CSV.

## Cómo usarlo
1. En Colab: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.
2. Sube el CSV `MEDIQA-CORR-2024-MS-TrainingData.csv` y ajusta `DATA_PATH`.
3. Con `QUICK_TEST = True` corre todo con unos 200 textos y una época, para comprobar que nada falla. Después pon `False`.
4. Qué se probó: el notebook completo corrió de punta a punta en CPU con transformers 5.22.1 usando modelos diminutos. Con los modelos reales se verificó la evaluación (BERTScore con deberta-xlarge-mnli). El entrenamiento de Bio_ClinicalBERT y Flan-T5 reales **no** se corrió (la máquina de prueba no tenía GPU): los tiempos y las métricas finales las verás al ejecutarlo en Colab con GPU.

## 0. Configuración

In [1]:
!pip install -q transformers datasets accelerate scikit-learn rouge rouge_score bert_score sentencepiece

In [2]:
import re
import random
import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_recall_fscore_support
from rouge import Rouge
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,
                          DataCollatorWithPadding, DataCollatorForSeq2Seq,
                          Trainer, TrainingArguments, Seq2SeqTrainer, Seq2SeqTrainingArguments)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_PATH = "MEDIQA-CORR-2024-MS-TrainingData.csv"   # sube el CSV a Colab o móntalo desde Drive
DETECTOR_NAME = "emilyalsentzer/Bio_ClinicalBERT"
CORRECTOR_NAME = "google/flan-t5-base"
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"     # el mismo del script oficial

QUICK_TEST = False        # True: ~200 textos y 1 época, solo para comprobar que el código corre
USE_BERTSCORE = True      # BERTScore descarga un modelo de 1,6 GB la primera vez
DUP_THRESHOLD = 0.7       # coseno TF-IDF a partir del cual dos textos son versiones del mismo caso
DEV_SIZE, TEST_SIZE = 0.15, 0.15
MAX_LEN_DETECTOR = 384    # tokens de (oración + contexto)
MAX_LEN_SOURCE = 384      # tokens de entrada del corrector
MAX_LEN_TARGET = 64       # tokens de la oración corregida

print("Dispositivo:", "GPU" if torch.cuda.is_available() else "CPU", "| transformers", transformers.__version__)

Dispositivo: GPU | transformers 5.22.1


## 1. Datos: carga, segmentación y agrupación

**Qué hace cada función y para qué sirve en la tarea**
- `load_data`: lee el CSV. En los textos sin error, las columnas de corrección vienen con el texto `"NA"`, igual que lo exige la tarea.
- `parse_sentences`: separa la columna `Sentences` en `(id, oración)`. La tarea evalúa los IDs de **esa** segmentación, así que no la reemplazamos por otra. Una oración puede tener saltos de línea dentro y su texto puede empezar con un número (p. ej. `37 C` en una tabla de signos vitales), así que el parser exige que el ID sea el siguiente en la secuencia (0, 1, 2, …). Verificado: las 1.219 oraciones erróneas del CSV coinciden con la segmentación.
- `make_case_groups`: agrupa los textos que son versiones del mismo caso de MedQA (similitud coseno TF-IDF > 0,7, componentes conexas). Sirve para que ninguna versión de un caso quede en train mientras otra está en evaluación.
- `describe_dataset`: resume el dataset (balance, largo, posición del error, grupos).

In [3]:
def load_data(path):
    """Carga el CSV, descarta la columna sin nombre y deja los tipos listos."""
    df = pd.read_csv(path, keep_default_na=False)          # "NA" se queda como texto "NA"
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    esperadas = ["Text ID", "Text", "Sentences", "Error Flag", "Error Sentence ID",
                 "Error Sentence", "Corrected Sentence", "Corrected Text"]
    faltan = [c for c in esperadas if c not in df.columns]
    assert not faltan, f"Faltan columnas: {faltan}"
    df["Error Flag"] = df["Error Flag"].astype(int)
    df["Error Sentence ID"] = df["Error Sentence ID"].astype(int)
    return df


def parse_sentences(sentences):
    """'0 Primera oración.\n1 Segunda...' -> [(0, 'Primera oración.'), (1, 'Segunda...')].

    Una línea abre la oración k solo si empieza con '<k> ' (k = siguiente ID esperado).
    Cualquier otra línea es continuación de la oración anterior.
    """
    out = []
    for line in sentences.split("\n"):
        prefijo = f"{len(out)} "
        if line.startswith(prefijo):
            out.append([len(out), line[len(prefijo):]])
        elif out:
            out[-1][1] += "\n" + line
        else:
            out.append([0, line])
    return [(i, t.strip()) for i, t in out]


def make_case_groups(df, threshold=DUP_THRESHOLD):
    """ID de grupo por texto: textos con coseno TF-IDF > threshold quedan en el mismo grupo."""
    X = TfidfVectorizer().fit_transform(df["Text"])
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    _, grupos = connected_components(sim > threshold, directed=False)
    return grupos


def describe_dataset(df):
    """Imprime y devuelve un resumen del dataset."""
    n_sent = df["Sentences"].apply(lambda s: len(parse_sentences(s)))
    err = df[df["Error Flag"] == 1]
    rel = err["Error Sentence ID"] / (n_sent[err.index] - 1)
    stats = {
        "textos": len(df),
        "con_error": int(df["Error Flag"].sum()),
        "sin_error": int((df["Error Flag"] == 0).sum()),
        "oraciones": int(n_sent.sum()),
        "oraciones_por_texto_media": round(float(n_sent.mean()), 1),
        "oraciones_por_texto_min_max": (int(n_sent.min()), int(n_sent.max())),
        "pct_oraciones_erroneas": round(100 * len(err) / n_sent.sum(), 2),
        "error_en_primera_pct": round(100 * float((err["Error Sentence ID"] == 0).mean()), 1),
        "error_en_ultima_pct": round(100 * float((err["Error Sentence ID"] == n_sent[err.index] - 1).mean()), 1),
        "error_en_ultimo_tercio_pct": round(100 * float((rel >= 2 / 3).mean()), 1),
        "palabras_por_texto_media": round(float(df["Text"].str.split().str.len().mean()), 1),
        "palabras_correccion_media": round(float(err["Corrected Sentence"].str.split().str.len().mean()), 1),
    }
    if "group" in df.columns:
        tam = df["group"].value_counts()
        stats["grupos"] = int(len(tam))
        stats["textos_con_gemelo_pct"] = round(100 * float((df["group"].map(tam) > 1).mean()), 1)
    for k, v in stats.items():
        print(f"{k:34s} {v}")
    return stats

In [4]:
# Prueba rápida
df = load_data(DATA_PATH)
assert len(df) == 2189 and df["Error Flag"].sum() == 1219
fila = df.iloc[0]
oraciones = parse_sentences(fila["Sentences"])
assert len(oraciones) == 8 and dict(oraciones)[fila["Error Sentence ID"]] == fila["Error Sentence"]
df["group"] = make_case_groups(df)
assert df.loc[0, "group"] == df.loc[1, "group"]   # ms-train-0 (con error) y ms-train-1 (correcto) son el mismo caso
stats = describe_dataset(df)

textos                             2189
con_error                          1219
sin_error                          970
oraciones                          27667
oraciones_por_texto_media          12.6
oraciones_por_texto_min_max        (2, 48)
pct_oraciones_erroneas             4.41
error_en_primera_pct               1.2
error_en_ultima_pct                62.5
error_en_ultimo_tercio_pct         85.5
palabras_por_texto_media           125.8
palabras_correccion_media          8.5
grupos                             1212
textos_con_gemelo_pct              88.7


## 2. Preparación: de textos a oraciones

**Qué hace cada función y para qué sirve en la tarea**
- `clean`: junta los espacios y saltos de línea; así los modelos reciben texto limpio.
- `build_sentence_table`: crea una fila por oración con su etiqueta `is_error`. El detector aprende a clasificar oraciones, y esta tabla es su dataset.
- `split_by_group` / `split_train_dev_test`: separan train, dev y test **por grupo de caso** (y estratificando por `Error Flag`). Partir solo por `Text ID` no basta: la versión correcta de un caso quedaría en train y la versión con error en evaluación, y el modelo podría memorizar la respuesta (con el split anterior, el 26 % de las correcciones de referencia de validación aparecían textuales en algún texto de entrenamiento).
- dev sirve para elegir el umbral del detector; test solo se usa para reportar.

In [5]:
def clean(s):
    """Normaliza espacios y saltos de línea."""
    return " ".join(str(s).split())


def build_sentence_table(df):
    """Una fila por oración: text_id, sent_id, sentence, context, n_sentences, rel_position, is_error."""
    filas = []
    for _, r in df.iterrows():
        oraciones = parse_sentences(r["Sentences"])
        n = len(oraciones)
        contexto = clean(r["Text"])
        for sid, texto in oraciones:
            filas.append({
                "text_id": r["Text ID"], "sent_id": sid, "sentence": clean(texto), "context": contexto,
                "n_sentences": n, "rel_position": sid / (n - 1) if n > 1 else 0.0,
                "is_error": int(r["Error Flag"] == 1 and sid == r["Error Sentence ID"]),
            })
    return pd.DataFrame(filas)


def split_by_group(df, size, seed=SEED):
    """Separa una fracción ~size de los GRUPOS, estratificando por 'Error Flag'. Devuelve (resto, parte)."""
    sgkf = StratifiedGroupKFold(n_splits=round(1 / size), shuffle=True, random_state=seed)
    idx_resto, idx_parte = next(sgkf.split(df, df["Error Flag"], groups=df["group"]))
    return df.iloc[idx_resto].reset_index(drop=True), df.iloc[idx_parte].reset_index(drop=True)


def split_train_dev_test(df, dev_size=DEV_SIZE, test_size=TEST_SIZE, seed=SEED):
    """train / dev / test sin grupos compartidos."""
    resto, test = split_by_group(df, test_size, seed)
    train, dev = split_by_group(resto, dev_size / (1 - test_size), seed)
    return train, dev, test

In [6]:
# Prueba rápida
if QUICK_TEST:   # se muestrean GRUPOS completos para no romper los pares
    g = df["group"].drop_duplicates().sample(110, random_state=SEED)
    df = df[df["group"].isin(g)].reset_index(drop=True)
train_df, dev_df, test_df = split_train_dev_test(df)
for a, b in [(train_df, dev_df), (train_df, test_df), (dev_df, test_df)]:
    assert set(a["group"]).isdisjoint(set(b["group"]))      # ningún caso compartido
    assert set(a["Text ID"]).isdisjoint(set(b["Text ID"]))

train_sent, dev_sent, test_sent = (build_sentence_table(d) for d in (train_df, dev_df, test_df))
assert train_sent.groupby("text_id")["is_error"].sum().isin([0, 1]).all()   # a lo sumo una oración errónea por texto
for nombre, d, s in [("train", train_df, train_sent), ("dev", dev_df, dev_sent), ("test", test_df, test_sent)]:
    print(f"{nombre:5s} {len(d):5d} textos ({100 * d['Error Flag'].mean():.1f} % con error) -> {len(s):6d} oraciones")

train  1563 textos (55.7 % con error) ->  19691 oraciones
dev     314 textos (55.7 % con error) ->   4246 oraciones
test    312 textos (55.8 % con error) ->   3730 oraciones


## 3. Evaluación y líneas base

**Qué hace cada función y para qué sirve en la tarea**
- `eval_flag_accuracy` y `eval_sentence_accuracy`: la métrica oficial de A y de B (porcentaje de textos acertados; en B, un texto sin error debe dar `-1`).
- `eval_correction`: ROUGE-1 F y BERTScore F1 entre la corrección generada y la de referencia, **configurados como el script oficial** (`rouge.Rouge`, `deberta-xlarge-mnli`, `rescale_with_baseline=True`, recorte a [0, 1]). Devuelve también un puntaje por texto, que se necesita para el Composite Score.
- `aggregate_score`: promedia las métricas de texto. El oficial usa ROUGE-1, BERTScore y BLEURT-20; aquí se omite BLEURT (instalación pesada en TensorFlow), así que el valor **no es comparable punto a punto** con la tabla del concurso. ROUGE-1 y BERTScore por separado sí lo son.
- `composite_score`: la regla del concurso para los textos sin error (1 si ambos son NA, 0 si solo uno lo es).
- **Líneas base:** definen el piso. Un modelo que no las supera no aporta.

In [7]:
def eval_flag_accuracy(y_true, y_pred):
    """Exactitud de la subtarea A."""
    return float(np.mean(np.array(y_true) == np.array(y_pred)))


def eval_sentence_accuracy(true_ids, pred_ids):
    """Exactitud de la subtarea B (un texto sin error se representa con -1)."""
    return float(np.mean(np.array(true_ids) == np.array(pred_ids)))


def aggregate_score(rouge1, bertscore=None, bleurt=None):
    """Promedio de las métricas de texto disponibles (el oficial usa las tres)."""
    valores = [v for v in (rouge1, bertscore, bleurt) if v is not None]
    return float(np.mean(valores))


_BERT_SCORER = None

def get_bert_scorer(model_type=BERTSCORE_MODEL):
    """BERTScorer cargado una sola vez.

    En transformers 5 el tokenizador de DeBERTa reporta model_max_length ~1e30 y bert_score lo pasa
    como max_length, lo que produce OverflowError. Se acota al largo real del modelo (512).
    """
    global _BERT_SCORER
    if _BERT_SCORER is None or _BERT_SCORER.model_type != model_type:
        from bert_score import BERTScorer
        _BERT_SCORER = BERTScorer(model_type=model_type, lang="en", rescale_with_baseline=True,
                                  device="cuda" if torch.cuda.is_available() else "cpu")
        if _BERT_SCORER._tokenizer.model_max_length > 100_000:
            _BERT_SCORER._tokenizer.model_max_length = 512
    return _BERT_SCORER


def release_bert_scorer():
    """Libera la memoria de BERTScore (~3 GB) antes de entrenar; se vuelve a cargar cuando haga falta."""
    global _BERT_SCORER
    _BERT_SCORER = None
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


_ROUGE = Rouge()

def rouge1_f(pred, ref):
    """ROUGE-1 F del paquete `rouge` (el del script oficial). Una predicción vacía puntúa 0 en vez de fallar."""
    try:
        return _ROUGE.get_scores(pred, ref)[0]["rouge-1"]["f"]
    except ValueError:            # "Hypothesis is empty" (p. ej. solo signos de puntuación)
        return 0.0


def _no_vacia(s):
    """bert_score falla con cadenas vacías en transformers 5; se reemplazan por '.' (puntaje ~0)."""
    s = clean(s)
    return s if s else "."


def eval_correction(preds, refs, bertscore=USE_BERTSCORE, bert_model=BERTSCORE_MODEL):
    """ROUGE-1 F (y BERTScore F1) entre correcciones generadas y de referencia, como el script oficial.

    Devuelve las medias, el AggregateScore (sin BLEURT) y 'per_text': el puntaje de cada par.
    """
    preds = [_no_vacia(p) for p in preds]
    refs = [_no_vacia(r) for r in refs]
    rouge = [rouge1_f(p, r) for p, r in zip(preds, refs)]
    out = {"rouge1": float(np.mean(rouge))}
    por_texto = list(rouge)
    if bertscore:
        _, _, f1 = get_bert_scorer(bert_model).score(preds, refs)
        f1 = np.clip(f1.numpy(), 0, 1)
        out["bertscore"] = float(f1.mean())
        por_texto = [(r + b) / 2 for r, b in zip(rouge, f1.tolist())]
    out["aggregate"] = aggregate_score(out["rouge1"], out.get("bertscore"))
    out["per_text"] = por_texto
    return out


def _es_na(x):
    return x is None or str(x).strip().upper() in ("", "NA", "NAN")


def composite_score(pred_correction, ref_correction, text_score=None):
    """1 si ambas son NA, 0 si solo una lo es, y el puntaje de texto si ninguna lo es."""
    if _es_na(pred_correction) and _es_na(ref_correction):
        return 1.0
    if _es_na(pred_correction) or _es_na(ref_correction):
        return 0.0
    return float(text_score)

In [8]:
# Prueba rápida
assert eval_flag_accuracy([1, 0, 1, 1], [1, 0, 0, 1]) == 0.75
assert eval_sentence_accuracy([6, -1, 2], [6, -1, 3]) == 2 / 3
assert composite_score("NA", "NA") == 1 and composite_score("NA", "algo") == 0
r = eval_correction(["the cat sat"], ["the cat sat down"], bertscore=False)
assert abs(r["rouge1"] - 6 / 7) < 1e-6                       # P = 3/3, R = 3/4 -> F = 6/7
r = eval_correction(["", "the cat sat"], ["a dog", "the cat sat"], bertscore=False)   # una predicción vacía no rompe nada
assert r["per_text"][0] == 0 and r["per_text"][1] > 0.99

### Líneas base
- `baseline_always_error`: responde siempre "hay error". Es el piso de A (55,7 % en el CSV completo).
- `baseline_never_error`: responde siempre "no hay error" (-1). En B acierta todos los textos correctos, así que su exactitud en B es la proporción de textos sin error (44,3 %). **Es el piso real de B.**
- `baseline_last_sentence`: responde siempre "hay error y está en la última oración". En B da ~35 % sobre todos los textos (62,5 % de aciertos en los textos con error × 55,7 % de textos con error), por debajo de "nunca hay error"; pero entre los textos con error es un piso fuerte de localización.
- `build_prompt_p1` y `parse_llm_output`: el prompt P#1 de MEDEC y su parser. Sirven para comparar con un LLM pequeño en Hugging Face (opcional).
- La línea base de C (copiar la oración errónea) está en la sección del corrector.

In [9]:
def baseline_always_error(df):
    """Predice siempre flag = 1."""
    return {"flag_acc": eval_flag_accuracy(df["Error Flag"], [1] * len(df))}


def baseline_never_error(df):
    """Predice siempre flag = 0 y oración -1."""
    return {"flag_acc": eval_flag_accuracy(df["Error Flag"], [0] * len(df)),
            "sentence_acc": eval_sentence_accuracy(df["Error Sentence ID"], [-1] * len(df))}


def baseline_last_sentence(df):
    """Predice siempre que hay error y que la errónea es la última oración."""
    n = df["Sentences"].apply(lambda s: len(parse_sentences(s)))
    pred = n - 1
    con_error = df["Error Flag"] == 1
    return {"sentence_acc_todos_los_textos": eval_sentence_accuracy(df["Error Sentence ID"], pred),
            "sentence_acc_solo_textos_con_error": eval_sentence_accuracy(df.loc[con_error, "Error Sentence ID"], pred[con_error])}


P1_INSTRUCCION = (
    "The following is a medical narrative about a patient. You are a skilled medical doctor reviewing the clinical text. "
    "The text is either correct or contains one error. The text has one sentence per line. Each line starts with the "
    "sentence ID, followed by a pipe character then the sentence to check. Check every sentence of the text. "
    "If the text is correct return the following output: CORRECT. If the text has a medical error related to treatment, "
    "management, cause, or diagnosis, return the sentence id of the sentence containing the error, followed by a space, "
    "and then a corrected version of the sentence. Finding and correcting the error requires medical knowledge and reasoning.")


def build_prompt_p1(sentences):
    """Prompt P#1 de MEDEC: una oración por línea con formato 'ID | oración'."""
    return P1_INSTRUCCION + "\n\n" + "\n".join(f"{i} | {clean(t)}" for i, t in sentences)


def parse_llm_output(output):
    """'CORRECT' -> (0, -1, 'NA'); '6 Corrected sentence' -> (1, 6, 'Corrected sentence')."""
    salida = output.strip()
    if salida.upper().startswith("CORRECT"):
        return 0, -1, "NA"
    m = re.match(r"^(\d+)\s+(.*)$", salida, flags=re.S)
    return (1, int(m.group(1)), m.group(2).strip()) if m else (0, -1, "NA")

In [10]:
# Prueba rápida y pisos (en test)
assert parse_llm_output("CORRECT") == (0, -1, "NA")
assert parse_llm_output("6 The pathogen was Streptococcus.") == (1, 6, "The pathogen was Streptococcus.")
print("Piso de A (siempre error):   ", baseline_always_error(test_df))
print("Piso de B (nunca error):     ", baseline_never_error(test_df))
print("Siempre la última oración:   ", baseline_last_sentence(test_df))

Piso de A (siempre error):    {'flag_acc': 0.5576923076923077}
Piso de B (nunca error):      {'flag_acc': 0.4423076923076923, 'sentence_acc': 0.4423076923076923}
Siempre la última oración:    {'sentence_acc_todos_los_textos': 0.3557692307692308, 'sentence_acc_solo_textos_con_error': 0.6379310344827587}


## 4. Detector (subtareas A y B)

**Qué hace cada función y para qué sirve en la tarea**
- `load_detector_tokenizer`: carga el tokenizador **sin pasar a minúsculas**. Bio_ClinicalBERT no trae `tokenizer_config.json` y Hugging Face asume `do_lower_case=True`, pero el modelo se preentrenó con texto en mayúsculas y minúsculas (`--do_lower_case=False` en el repositorio de los autores). Con minúsculas, "COPD" se vuelve `cop ##d`.
- `tokenize_sentence_with_context`: convierte `(oración, texto completo)` en ids de tokens. Se entrega el texto completo como segundo segmento, porque muchos errores solo se ven por contradicción con otras oraciones. Se trunca solo el contexto, nunca la oración (afecta al ~2 % de los textos, que pasan de 384 tokens).
- `compute_class_weights`: pesa más la clase "error" (4,4 % de las oraciones). Sin esto el modelo aprende a decir siempre "correcta".
- `WeightedTrainer`: el `Trainer` de Hugging Face, pero con pérdida de entropía cruzada ponderada.
- `train_error_classifier`: afina Bio_ClinicalBERT y reporta precisión, recall y F1 de la clase error (a nivel de oración, en dev).
- `predict_sentence_scores`: probabilidad de error de cada oración de un texto.
- `choose_threshold`, `decide_flag`, `pick_error_sentence`: convierten probabilidades de oración en las salidas de la tarea. Flag = 1 si la oración más sospechosa pasa el umbral, y B es esa oración. El umbral se elige en **dev**.

In [11]:
def load_detector_tokenizer(model_name=DETECTOR_NAME):
    """Tokenizador del detector respetando mayúsculas (Bio_ClinicalBERT es cased)."""
    return AutoTokenizer.from_pretrained(model_name, do_lower_case=False)


def tokenize_sentence_with_context(batch, tokenizer, max_length=MAX_LEN_DETECTOR):
    """Tokeniza el par (oración, contexto); trunca solo el contexto."""
    return tokenizer(batch["sentence"], batch["context"], truncation="only_second", max_length=max_length)


def compute_class_weights(labels):
    """Peso de cada clase = n / (2 * conteo de la clase). La clase minoritaria pesa más."""
    labels = np.array(labels)
    pesos = [len(labels) / (2 * max((labels == c).sum(), 1)) for c in (0, 1)]
    return torch.tensor(pesos, dtype=torch.float)


class WeightedTrainer(Trainer):
    """Trainer con entropía cruzada ponderada por clase."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def train_error_classifier(train_sent_df, val_sent_df, model_name=DETECTOR_NAME, output_dir="detector",
                           epochs=3, lr=2e-5, batch_size=16):
    """Afina el detector. Devuelve (trainer, tokenizer)."""
    tokenizer = load_detector_tokenizer(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    def preparar(sdf):
        ds = Dataset.from_pandas(sdf[["sentence", "context", "is_error"]].rename(columns={"is_error": "label"}),
                                 preserve_index=False)
        return ds.map(lambda b: tokenize_sentence_with_context(b, tokenizer), batched=True,
                      remove_columns=["sentence", "context"])

    def metricas(eval_pred):
        logits, labels = eval_pred
        p, r, f1, _ = precision_recall_fscore_support(labels, logits.argmax(-1), average="binary", zero_division=0)
        return {"precision": p, "recall": r, "f1": f1}

    args = TrainingArguments(output_dir=output_dir, num_train_epochs=epochs, learning_rate=lr,
                             per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
                             eval_strategy="epoch", save_strategy="no", logging_steps=100,
                             fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
    trainer = WeightedTrainer(model=model, args=args, train_dataset=preparar(train_sent_df),
                              eval_dataset=preparar(val_sent_df), processing_class=tokenizer,
                              data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metricas,
                              class_weights=compute_class_weights(train_sent_df["is_error"]))
    trainer.train()
    return trainer, tokenizer

### Inferencia de A y B

In [12]:
def predict_sentence_scores(sentences, context, model, tokenizer):
    """Probabilidad de error de cada oración de un texto. sentences = [(id, oración), ...]."""
    textos = [clean(t) for _, t in sentences]
    enc = tokenizer(textos, [context] * len(textos), truncation="only_second", max_length=MAX_LEN_DETECTOR,
                    padding=True, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1)[:, 1].tolist()


def decide_flag(scores, threshold):
    """Flag del texto = 1 si la oración más sospechosa supera el umbral."""
    return int(max(scores) >= threshold)


def pick_error_sentence(scores, flag):
    """ID de la oración más sospechosa si hay error, y -1 si no."""
    return int(np.argmax(scores)) if flag == 1 else -1


def choose_threshold(scores_per_text, flags_true):
    """Umbral (malla de 0.05 a 0.99) que maximiza la exactitud del flag en dev."""
    mejor_umbral, mejor_acc = 0.5, -1.0
    for u in list(np.arange(0.05, 0.96, 0.05)) + [0.97, 0.98, 0.99]:   # con pesos de clase las probabilidades quedan infladas
        acc = eval_flag_accuracy(flags_true, [decide_flag(s, u) for s in scores_per_text])
        if acc > mejor_acc:
            mejor_umbral, mejor_acc = float(u), acc
    return mejor_umbral

## 5. Corrector (subtarea C)

**Qué hace cada función y para qué sirve en la tarea**
- `build_correction_source`: arma la entrada del modelo: la oración errónea más el texto completo como contexto, con una instrucción al inicio (Flan-T5 fue entrenado con instrucciones). Se usa igual al entrenar y al predecir.
- `build_correction_pairs`: crea las parejas `source -> target` con los textos que tienen error.
- `tokenize_for_seq2seq`: convierte `source` y `target` a tokens; los tokens del `target` son los `labels`.
- `train_corrector`: afina Flan-T5 con `Seq2SeqTrainer` (sin fp16: T5 desborda en media precisión).
- `generate_correction`: genera la oración corregida con búsqueda por haz (beam search). Si el modelo devuelve una cadena vacía, se devuelve la oración original (la línea base de copiar), que nunca puntúa peor que vacío.
- `baseline_copy_sentence`: línea base de C. Copiar la oración errónea ya da un ROUGE alto, porque la corrección cambia una o dos palabras.

In [13]:
def build_correction_source(sentence, context):
    """Entrada del corrector: instrucción + oración errónea + texto completo como contexto."""
    return f"Correct the medical error in this sentence. Sentence: {clean(sentence)} Context: {clean(context)}"


def build_correction_pairs(df):
    """Tabla (text_id, source, target) con los textos que tienen error."""
    con_error = df[df["Error Flag"] == 1]
    return pd.DataFrame({
        "text_id": con_error["Text ID"].values,
        "source": [build_correction_source(s, t) for s, t in zip(con_error["Error Sentence"], con_error["Text"])],
        "target": [clean(c) for c in con_error["Corrected Sentence"]],
    })


def tokenize_for_seq2seq(batch, tokenizer, max_input=MAX_LEN_SOURCE, max_target=MAX_LEN_TARGET):
    """Tokeniza source y target; los ids del target son los 'labels'."""
    enc = tokenizer(batch["source"], max_length=max_input, truncation=True)
    enc["labels"] = tokenizer(text_target=batch["target"], max_length=max_target, truncation=True)["input_ids"]
    return enc


def train_corrector(train_pairs, val_pairs, model_name=CORRECTOR_NAME, output_dir="corrector",
                    epochs=6, lr=1e-4, batch_size=8):
    """Afina el corrector. Devuelve (trainer, tokenizer)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def preparar(pairs):
        ds = Dataset.from_pandas(pairs[["source", "target"]], preserve_index=False)
        return ds.map(lambda b: tokenize_for_seq2seq(b, tokenizer), batched=True, remove_columns=["source", "target"])

    args = Seq2SeqTrainingArguments(output_dir=output_dir, num_train_epochs=epochs, learning_rate=lr,
                                    per_device_train_batch_size=batch_size, per_device_eval_batch_size=16,
                                    eval_strategy="epoch", save_strategy="no", logging_steps=50,
                                    report_to="none", seed=SEED)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=preparar(train_pairs),
                             eval_dataset=preparar(val_pairs), processing_class=tokenizer,
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100))
    trainer.train()
    return trainer, tokenizer


def generate_correction(source, model, tokenizer, num_beams=4, max_new_tokens=MAX_LEN_TARGET, fallback=None):
    """Genera la oración corregida a partir de la entrada del corrector.

    fallback: texto a devolver si la generación sale vacía (normalmente, la oración errónea).
    """
    enc = tokenizer(source, return_tensors="pt", truncation=True, max_length=MAX_LEN_SOURCE).to(model.device)
    model.eval()
    with torch.no_grad():
        ids = model.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens)
    salida = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
    return salida if salida else clean(fallback or "")


def baseline_copy_sentence(df):
    """Línea base de C: devolver la oración errónea sin cambios."""
    con_error = df[df["Error Flag"] == 1]
    preds = [clean(s) for s in con_error["Error Sentence"]]
    refs = [clean(s) for s in con_error["Corrected Sentence"]]
    r = eval_correction(preds, refs)
    r.pop("per_text")
    return r

In [14]:
# Prueba rápida y piso de C (en test)
train_pairs, dev_pairs = build_correction_pairs(train_df), build_correction_pairs(dev_df)
assert len(train_pairs) == int(train_df["Error Flag"].sum())
print(train_pairs.iloc[0]["source"][:200], "\n->", train_pairs.iloc[0]["target"])
print("Piso de C (copiar la oración errónea):", baseline_copy_sentence(test_df))
release_bert_scorer()   # libera memoria antes de entrenar

Correct the medical error in this sentence. Sentence: After reviewing imaging, the causal pathogen was determined to be Haemophilus influenzae. Context: A 53-year-old man comes to the physician becaus 
-> After reviewing imaging, the causal pathogen was determined to be Streptococcus pneumoniae.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.04GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.04GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

Piso de C (copiar la oración errónea): {'rouge1': 0.5997639032922741, 'bertscore': 0.6732084155082703, 'aggregate': 0.6364861594002722}


## 6. Pipeline completo

**Qué hace cada función y para qué sirve en la tarea**
- `run_pipeline`: aplica detector y corrector a cada texto. Si el flag es 0, la salida es `NA` y no se llama al corrector.
- `evaluate_all`: calcula A y B (exactitud), C (solo sobre pares donde sistema y referencia tienen corrección, como el oficial) y el Composite Score promedio.
- `write_predictions`: guarda las predicciones en CSV y en el formato de envío del concurso (`text_id flag sent_id corrección`, una línea por texto).

In [15]:
def run_pipeline(df, detector, det_tokenizer, corrector, cor_tokenizer, threshold):
    """Predice A, B y C para cada texto de df. Devuelve un DataFrame con una fila por texto."""
    filas = []
    for _, r in tqdm(df.iterrows(), total=len(df)):
        oraciones = parse_sentences(r["Sentences"])
        scores = predict_sentence_scores(oraciones, clean(r["Text"]), detector, det_tokenizer)
        flag = decide_flag(scores, threshold)
        sid = pick_error_sentence(scores, flag)
        if flag == 1:
            fuente = build_correction_source(dict(oraciones)[sid], r["Text"])
            correccion = generate_correction(fuente, corrector, cor_tokenizer, fallback=dict(oraciones)[sid])
        else:
            correccion = "NA"
        filas.append({"Text ID": r["Text ID"], "Error Flag": flag, "Error Sentence ID": sid,
                      "Corrected Sentence": correccion, "max_score": max(scores)})
    return pd.DataFrame(filas)


def evaluate_all(pred_df, gold_df):
    """Métricas de A, B y C, más el Composite Score promedio."""
    m = gold_df.merge(pred_df, on="Text ID", suffixes=("", "_pred"))
    res = {"flag_acc": eval_flag_accuracy(m["Error Flag"], m["Error Flag_pred"]),
           "sentence_acc": eval_sentence_accuracy(m["Error Sentence ID"], m["Error Sentence ID_pred"])}
    ambos = m[(m["Error Flag"] == 1) & (m["Error Flag_pred"] == 1)]
    por_texto = {}
    if len(ambos):
        c = eval_correction([clean(x) for x in ambos["Corrected Sentence_pred"]],
                            [clean(x) for x in ambos["Corrected Sentence"]])
        por_texto = dict(zip(ambos["Text ID"], c.pop("per_text")))
        res.update(c)
    res["pares_de_correccion"] = len(ambos)
    res["composite"] = float(np.mean([composite_score(r["Corrected Sentence_pred"], r["Corrected Sentence"],
                                                      por_texto.get(r["Text ID"]))
                                      for _, r in m.iterrows()]))
    return res


def write_predictions(pred_df, path):
    """Guarda las predicciones en CSV y en el formato de envío oficial (.txt)."""
    pred_df.drop(columns=["max_score"], errors="ignore").to_csv(path, index=False)
    with open(re.sub(r"\.csv$", "", path) + ".txt", "w", encoding="utf-8") as f:
        for _, r in pred_df.iterrows():
            correccion = "NA" if r["Error Flag"] == 0 else (clean(r["Corrected Sentence"]) or "NA")
            f.write(f'{r["Text ID"]} {r["Error Flag"]} {r["Error Sentence ID"]} {correccion}\n')

## 7. Ejecución completa

Orden: entrenar el detector, elegir el umbral en **dev**, entrenar el corrector, correr el pipeline en **test** y evaluar. Test no se toca hasta este punto.

In [16]:
# 7.1 Detector (dev se usa solo para ver las métricas por época; no se guarda el "mejor" checkpoint)
det_trainer, det_tok = train_error_classifier(train_sent, dev_sent, epochs=1 if QUICK_TEST else 3)
detector = det_trainer.model

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/19691 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/4246 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.253271,0.351011,0.552419,0.782857,0.647754
2,0.129265,0.204128,0.520900,0.925714,0.666667
3,0.160632,0.295868,0.539249,0.902857,0.675214


In [17]:
# 7.2 Umbral en dev; A y B se reportan en test
def scores_por_texto(d):
    return [predict_sentence_scores(parse_sentences(r["Sentences"]), clean(r["Text"]), detector, det_tok)
            for _, r in tqdm(d.iterrows(), total=len(d))]

dev_scores = scores_por_texto(dev_df)
threshold = choose_threshold(dev_scores, dev_df["Error Flag"])
print("Umbral elegido en dev:", threshold)

test_scores = scores_por_texto(test_df)
flags = [decide_flag(s, threshold) for s in test_scores]
ids = [pick_error_sentence(s, f) for s, f in zip(test_scores, flags)]
print("A (flag) exactitud en test:", round(eval_flag_accuracy(test_df["Error Flag"], flags), 4),
      "| piso:", round(baseline_always_error(test_df)["flag_acc"], 4))
print("B (oración) exactitud en test:", round(eval_sentence_accuracy(test_df["Error Sentence ID"], ids), 4),
      "| piso:", round(baseline_never_error(test_df)["sentence_acc"], 4))
p, rc, f1, _ = precision_recall_fscore_support(test_df["Error Flag"], flags, average="binary", zero_division=0)
print(f"Flag a nivel de texto -> precisión {p:.3f}, recall {rc:.3f}, F1 {f1:.3f}")
con_error = (test_df["Error Flag"] == 1).values
print("Localización si se sabe que hay error (argmax sobre textos con error):",
      round(float(np.mean([int(np.argmax(s)) == t for s, t, c in zip(test_scores, test_df["Error Sentence ID"], con_error) if c])), 4))

  0%|          | 0/314 [00:00<?, ?it/s]

Umbral elegido en dev: 0.55


  0%|          | 0/312 [00:00<?, ?it/s]

A (flag) exactitud en test: 0.5417 | piso: 0.5577
B (oración) exactitud en test: 0.5353 | piso: 0.4423
Flag a nivel de texto -> precisión 0.558, recall 0.856, F1 0.676
Localización si se sabe que hay error (argmax sobre textos con error): 0.9713


In [18]:
# 7.3 Corrector
cor_trainer, cor_tok = train_corrector(train_pairs, dev_pairs, epochs=1 if QUICK_TEST else 6)
corrector = cor_trainer.model

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/870 [00:00<?, ? examples/s]

Map:   0%|          | 0/175 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,1.572673,1.351871
2,1.403104,1.311117
3,1.154612,1.308579
4,1.070005,1.313850
5,0.997174,1.312329
6,0.931051,1.317739


In [19]:
# 7.4 Corrector solo, con la oración errónea REAL (cota superior de C) frente a copiar la oración, en test
ver = test_df[test_df["Error Flag"] == 1]
preds = [generate_correction(build_correction_source(s, t), corrector, cor_tok, fallback=s)
         for s, t in tqdm(zip(ver["Error Sentence"], ver["Text"]), total=len(ver))]
refs = [clean(s) for s in ver["Corrected Sentence"]]
res_c = eval_correction(preds, refs); res_c.pop("per_text")
print("Corrector (oración real):", res_c)
print("Copiar la oración:       ", baseline_copy_sentence(test_df))
print("Correcciones idénticas a la referencia:", round(float(np.mean([p == r for p, r in zip(preds, refs)])), 4))

  0%|          | 0/174 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

Corrector (oración real): {'rouge1': 0.5665880058397323, 'bertscore': 0.6390079259872437, 'aggregate': 0.602797965913488}


Copiar la oración:        {'rouge1': 0.5997639032922741, 'bertscore': 0.6732084155082703, 'aggregate': 0.6364861594002722}
Correcciones idénticas a la referencia: 0.0


In [20]:
# 7.5 Pipeline completo en test
pred_test = run_pipeline(test_df, detector, det_tok, corrector, cor_tok, threshold)
resultados = evaluate_all(pred_test, test_df)
print(resultados)
write_predictions(pred_test, "predicciones_test.csv")   # también escribe predicciones_test.txt (formato oficial)

  0%|          | 0/312 [00:00<?, ?it/s]

{'flag_acc': 0.5416666666666666, 'sentence_acc': 0.5352564102564102, 'rouge1': 0.5619404384661193, 'bertscore': 0.6377133131027222, 'aggregate': 0.5998268757844207, 'pares_de_correccion': 149, 'composite': 0.3505583665642899}


## 8. Preguntas para el informe
1. ¿Qué piso es más difícil de superar en A y en B, y por qué? (Ojo: en B el piso es "nunca hay error", no "siempre la última".)
2. ¿Qué umbral eligió `choose_threshold` en dev y qué pasa con precisión y recall en test al moverlo?
3. ¿Cuánto mejora el corrector frente a copiar la oración? ¿Qué dicen ROUGE-1 y BERTScore?
4. ¿Cuánto cae C cuando el detector se equivoca de oración (7.4 frente a 7.5)?
5. ¿Cuánto cambiarían las métricas si se partiera por `Text ID` en lugar de por grupo? ¿Por qué?
6. ¿Qué cambiarías con más tiempo: pesos de clase más suaves, otro modelo base, más contexto, más épocas?
7. ¿Qué haría diferente WangLab y qué costo tendría replicarlo (recuperación TF-IDF en MedQA más GPT-4 con DSPy)?